   # 03 – Road Distances
   Goal: get road distances (miles) from each factory to each customer ZIP code using the OpenRouteService Matrix API with the **driving-hgv** (truck) profile, and save them for the cost model.

In [6]:
import numpy as np
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
ORS_KEY = os.getenv("ORS_API_KEY")
URL = "https://api.heigit.org/openrouteservice/v2/matrix/driving-hgv"

factories = pd.read_csv("../data/processed/factories.csv")
zips = pd.read_csv("../data/processed/zips.csv", dtype={"zip": str})

# Test: first factory -> first ZIP code. ORS expects [longitude, latitude]!
origin = [factories.loc[0, "longitude"], factories.loc[0, "latitude"]]
destination = [zips.loc[0, "lng"], zips.loc[0, "lat"]]

response = requests.post(
    URL,
    json={"locations": [origin, destination], "sources": [0], "destinations": [1],
          "metrics": ["distance"], "units": "mi"},
    headers={"Authorization": ORS_KEY},
)
print("Status:", response.status_code)
print(factories.loc[0, "factory"], "→", zips.loc[0, "city"], zips.loc[0, "state_id"])
print(response.json())

Status: 200
Lot's O' Nuts → Holyoke MA
{'distances': [[2639.15]], 'destinations': [{'location': [-72.640832, 42.212364], 'snapped_distance': 26.77}], 'sources': [{'location': [-111.768019, 32.881032], 'snapped_distance': 95.71}], 'metadata': {'attribution': 'openrouteservice.org | OpenStreetMap contributors', 'service': 'matrix', 'timestamp': 1793739400667, 'query': {'locations': [[-111.768036, 32.881893], [-72.64109, 42.21251]], 'profile': 'driving-hgv', 'profileName': 'driving-hgv', 'responseType': 'json', 'sources': ['0'], 'destinations': ['1'], 'metrics': ['distance'], 'units': 'mi'}, 'engine': {'version': '9.10.0', 'build_date': '2026-08-13T11:53:14Z', 'graph_date': '2026-09-23T04:04:43Z', 'osm_date': '2026-09-13T23:59:59Z'}}}


In [7]:
import time

factory_coords = factories[["longitude", "latitude"]].values.tolist()
results = []
BATCH = 100

for start in range(0, len(zips), BATCH):
    batch = zips.iloc[start:start + BATCH]
    zip_coords = batch[["lng", "lat"]].values.tolist()

    response = requests.post(
        URL,
        json={
            "locations": factory_coords + zip_coords,
            "sources": list(range(len(factory_coords))),
            "destinations": list(range(len(factory_coords), len(factory_coords) + len(zip_coords))),
            "metrics": ["distance"],
            "units": "mi",
        },
        headers={"Authorization": ORS_KEY},
    )
    response.raise_for_status()
    distances = response.json()["distances"]

    for i, factory in enumerate(factories["factory"]):
        for j, zip_code in enumerate(batch["zip"]):
            results.append({"factory": factory, "zip": zip_code, "road_miles": distances[i][j]})

    print(f"ZIP codes {start + 1}–{start + len(batch)} done")
    time.sleep(2)

road_distances = pd.DataFrame(results)
print("Rows:", len(road_distances), "| Missing distances:", road_distances["road_miles"].isna().sum())

ZIP codes 1–100 done
ZIP codes 101–200 done
ZIP codes 201–300 done
ZIP codes 301–400 done
ZIP codes 401–500 done
ZIP codes 501–600 done
ZIP codes 601–631 done
Rows: 3155 | Missing distances: 20


In [8]:
missing = road_distances[road_distances["road_miles"].isna()]
missing_zips = missing["zip"].unique()

print("Missing routes per factory:\n", missing["factory"].value_counts(), "\n")
display(zips[zips["zip"].isin(missing_zips)])

Missing routes per factory:
 factory
Lot's O' Nuts        4
Wicked Choccy's      4
Sugar Shack          4
Secret Factory       4
The Other Factory    4
Name: count, dtype: int64 



,zip,lat,lng,city,state_id
468,80538,40.48174,-105.16825,Loveland,CO
475,83501,46.22830,-116.90608,Lewiston,ID
483,84062,40.38586,-111.68421,Pleasant Grove,UT
488,84604,40.33130,-111.57284,Provo,UT


## Missing routes
4 ZIP codes (Loveland CO, Lewiston ID, Pleasant Grove UT, Provo UT) returned no distance from the Matrix API – their ZIP points are too far from a truck-accessible road for the default snapping radius. These 20 routes are requested individually from the Directions API with an unlimited snapping radius.

In [9]:
DIRECTIONS_URL = "https://api.heigit.org/openrouteservice/v2/directions/driving-hgv"
factory_lookup = factories.set_index("factory")
zip_lookup = zips.set_index("zip")

for idx, row in road_distances[road_distances["road_miles"].isna()].iterrows():
    f = factory_lookup.loc[row["factory"]]
    z = zip_lookup.loc[row["zip"]]

    response = requests.post(
        DIRECTIONS_URL,
        json={
            "coordinates": [[f["longitude"], f["latitude"]], [z["lng"], z["lat"]]],
            "radiuses": [-1, -1],
            "units": "mi",
        },
        headers={"Authorization": ORS_KEY},
    )
    if response.status_code == 200:
        road_distances.loc[idx, "road_miles"] = response.json()["routes"][0]["summary"]["distance"]
    else:
        print(row["factory"], "→", row["zip"], "failed:", response.status_code, response.text[:200])
    time.sleep(2)

print("Missing distances after fix:", road_distances["road_miles"].isna().sum())

Missing distances after fix: 0


## Validation
Road distance should be longer than the straight-line distance, typically by 10–40% in the US. Routes with an unusual ratio are flagged.

In [10]:
def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 3958.8 * 2 * np.arcsin(np.sqrt(a))

check = road_distances.merge(factories, on="factory").merge(zips, on="zip")
check["straight_miles"] = haversine_miles(check["latitude"], check["longitude"], check["lat"], check["lng"])
check["ratio"] = check["road_miles"] / check["straight_miles"]

print(check["ratio"].describe(), "\n")
display(check.sort_values("ratio")[["factory", "city", "state_id", "straight_miles", "road_miles", "ratio"]].head(5))
display(check.sort_values("ratio")[["factory", "city", "state_id", "straight_miles", "road_miles", "ratio"]].tail(5))

count    3155.000000
mean        1.195930
std         0.068940
min         1.051681
25%         1.148502
50%         1.185060
75%         1.230549
max         1.602074
Name: ratio, dtype: float64 



,factory,city,state_id,straight_miles,road_miles,ratio
2500,Lot's O' Nuts,Tucson,AZ,61.834363,65.03,1.051681
2098,Lot's O' Nuts,Yuma,AZ,168.218348,177.90,1.057554
632,Wicked Choccy's,Florence,SC,164.151605,173.81,1.058838
985,The Other Factory,Decatur,AL,171.660803,181.80,1.059065
643,Wicked Choccy's,Macon,GA,160.044810,170.24,1.063702


,factory,city,state_id,straight_miles,road_miles,ratio
1716,Sugar Shack,Rapid City,SD,435.001651,674.42,1.550385
1717,Sugar Shack,Fargo,ND,92.240890,144.75,1.569261
2090,Lot's O' Nuts,Mesa,AZ,35.585847,55.89,1.570568
1715,Sugar Shack,Aberdeen,SD,213.792431,336.05,1.571852
1387,Secret Factory,Burlington,IA,49.741779,79.69,1.602074


In [11]:
road_distances.to_csv("../data/processed/road_distances.csv", index=False)
print("Saved:", len(road_distances), "routes")

Saved: 3155 routes
